# First Time Running The Current Balance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
import math

def _to_int_like(value):
    """
    Mimics Excel TEXT(number, "000..."):
    - Accepts ints, floats, and numeric strings.
    - Treats NaN/None as None.
    - Floats are truncated like Excel formatting without decimals.
    """
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None

    if isinstance(value, str):
        s = value.strip()
        if s.isdigit():
            return int(s)
        try:
            return int(float(s))  # e.g., "123.0" -> 123
        except ValueError:
            return None

    if isinstance(value, (int, float)):
        return int(value)

    return None


def format_encounter_id(facility_code, encounter_id):
    """
    Python equivalent of your Excel formula for:
      A: Facility code
      B: EncounterID

    Rules:
      - If TRIM(A) starts with "Q" -> pad to 7
      - If A in {BACC, BOLE, BOMC, BPHC, SJMA, SJMC, SJRD} -> pad to 13
      - If A in {SJPK, SJPR} -> pad to 11
      - If A == SJOK -> pad to 9
      - If A == LPAZ -> pad to 8
      - If A == MIWI -> pad to 12
      - If A == MEWI and LEFT(TRIM(A),1) <> "R" -> pad to 9  (true for "MEWI")
      - If A == SMMC -> pad to 12
      - Else -> return EncounterID as-is
    """
    a = "" if facility_code is None else str(facility_code).strip()

    pad_len = None
    if a[:1].upper() == "Q":
        pad_len = 7
    elif a in {"BACC", "BOLE", "BOMC", "BPHC", "SJMA", "SJMC", "SJRD"}:
        pad_len = 13
    elif a in {"SJPK", "SJPR"}:
        pad_len = 11
    elif a == "SJOK":
        pad_len = 9
    elif a == "LPAZ":
        pad_len = 8
    elif a == "MIWI":
        pad_len = 12
    elif a == "MEWI" and (a[:1].upper() != "R"):
        pad_len = 9
    elif a == "SMMC":
        pad_len = 12
    else:
        pad_len = None

    if pad_len is None:
        return encounter_id

    n = _to_int_like(encounter_id)
    if n is None:
        # Excel TEXT would error on non-numeric; here we pass original through.
        # Change to: raise ValueError(...) if you want strict behavior.
        return encounter_id

    return str(abs(n)).zfill(pad_len)

In [3]:
import pandas as pd
import time

start_time = time.time()

rec_vs_work = pd.read_excel(
    r"C:\Users\IN10052060\Desktop\Received VS Worked Master File.xlsx",
    sheet_name='UniqeData'
)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"File loaded in {elapsed_time:.2f} seconds")

File loaded in 67.53 seconds


In [4]:
rec_vs_work.head()

,Unique,Drill TYPE,Project Name,FacilityGroupName,FacilityCode,EncounterID,Assigned Y/N,Status,Final Status,Date,...,Amt3,Grp_Cd4,Rsn_Cd4,Amt4,Grp_Cd5,Rsn_Cd5,Amt5,Grp_Cd6,Rsn_Cd6,Amt6
0,FNWI40013277560,Contractual,CO-253 Sequestration,Wheaton,FNWI,40013277560,NaN,N,NaN,6/12/2026-3:28:59 PM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GRMC10009536020,Contractual,CO-253 Sequestration,Flint,GRMC,10009536020,NaN,N,NaN,6/12/2026-3:28:59 PM,...,6.92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EKWI31002766923,Contractual,CO-253 Sequestration,Milwaukee,EKWI,31002766923,NaN,N,NaN,6/12/2026-3:28:59 PM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FNWI40013180146,Contractual,CO-253 Sequestration,Wheaton,FNWI,40013180146,NaN,N,NaN,6/12/2026-3:28:59 PM,...,33.23,CO,94,-24.55,NaN,NaN,NaN,NaN,NaN,NaN
4,FNWI40013208703,Contractual,CO-253 Sequestration,Wheaton,FNWI,40013208703,NaN,N,NaN,6/12/2026-3:28:59 PM,...,-1.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
rec_vs_work = rec_vs_work.dropna(subset = 'Unique')

In [6]:
rec_vs_work.shape

(139706, 82)

In [7]:
col = ['FacilityCode', 'EncounterID']

In [8]:
df = rec_vs_work[col]

In [9]:
df.loc[:, 'FacilityCode'] = df['FacilityCode'].str.strip()

In [10]:
df = df.copy()

In [11]:
df["EncounterID_formatted"] = df.apply(
    lambda r: format_encounter_id(r["FacilityCode"], r["EncounterID"]),
    axis=1
)

In [12]:
df["Query Code"] = "insert into #act values('" + df["FacilityCode"].astype(str) + "','" + df["EncounterID_formatted"].astype(str) + "')"

In [13]:
df.shape

(139706, 4)

In [14]:
df.head()

,FacilityCode,EncounterID,EncounterID_formatted,Query Code
0,FNWI,40013277560,40013277560,"insert into #act values('FNWI','40013277560')"
1,GRMC,10009536020,10009536020,"insert into #act values('GRMC','10009536020')"
2,EKWI,31002766923,31002766923,"insert into #act values('EKWI','31002766923')"
3,FNWI,40013180146,40013180146,"insert into #act values('FNWI','40013180146')"
4,FNWI,40013208703,40013208703,"insert into #act values('FNWI','40013208703')"


In [15]:
sql_db = pd.read_excel(r'C:\Users\IN10052060\Desktop\Python\SqlDBStructure.xlsx', sheet_name = 'All Sites')

In [16]:
sql_db['Facility Code'] = sql_db['Facility Code'].str.strip()

In [17]:
sql_db.head()

,Facility Code,Data Bases,Server Name,Site Name
0,A116,TranA116,AHS-A2RSAS03.extapp.local,IMH
1,A118,TranA118,AHS-A2RSAS03.extapp.local,IMH
2,A120,TranA120,AHS-A2RSAS03.extapp.local,IMH
3,A122,TranA122,AHS-A2RSAS03.extapp.local,IMH
4,A124,TranA124,AHS-A2RSAS03.extapp.local,IMH


In [18]:
df_total = df.copy()

In [19]:
df_total = df_total.merge(sql_db, how = 'left', left_on = 'FacilityCode', right_on = 'Facility Code')
df_total.head()

,FacilityCode,EncounterID,EncounterID_formatted,Query Code,Facility Code,Data Bases,Server Name,Site Name
0,FNWI,40013277560,40013277560,"insert into #act values('FNWI','40013277560')",FNWI,TranFNWI,ALPTRAN28.accretivehealth.local,Wheaton
1,GRMC,10009536020,10009536020,"insert into #act values('GRMC','10009536020')",GRMC,TranGRMC,AHS-TRAN11.accretivehealth.local,Flint
2,EKWI,31002766923,31002766923,"insert into #act values('EKWI','31002766923')",EKWI,TranEKWI,AHS-TRAN14.accretivehealth.local,Milwaukee
3,FNWI,40013180146,40013180146,"insert into #act values('FNWI','40013180146')",FNWI,TranFNWI,ALPTRAN28.accretivehealth.local,Wheaton
4,FNWI,40013208703,40013208703,"insert into #act values('FNWI','40013208703')",FNWI,TranFNWI,ALPTRAN28.accretivehealth.local,Wheaton


In [20]:
df_total.drop(['Facility Code', 'Site Name'], axis = 1, inplace = True)

In [21]:
df_total.head(2)

,FacilityCode,EncounterID,EncounterID_formatted,Query Code,Data Bases,Server Name
0,FNWI,40013277560,40013277560,"insert into #act values('FNWI','40013277560')",TranFNWI,ALPTRAN28.accretivehealth.local
1,GRMC,10009536020,10009536020,"insert into #act values('GRMC','10009536020')",TranGRMC,AHS-TRAN11.accretivehealth.local


In [22]:
df_total['Server Name'].unique()

array(['ALPTRAN28.accretivehealth.local',
       'AHS-TRAN11.accretivehealth.local',
       'AHS-TRAN14.accretivehealth.local',
       'ALPTRAN27.accretivehealth.local',
       'AHS-TRAN18.accretivehealth.local',
       'AHS-TRAN10.accretivehealth.local',
       'AHS-TRAN12.accretivehealth.local',
       'AHS-Tran11.accretivehealth.local',
       'AHS-TRAN15.accretivehealth.local',
       'ALPTRAN25.accretivehealth.local',
       'ALPTRAN31.accretivehealth.local',
       'ALPTRAN29.accretivehealth.local',
       'ALPTRAN26.accretivehealth.local', 'ALPA2ATRN42.extapp.local',
       'ALPTRAN33.accretivehealth.local',
       'ALPTRAN24.accretivehealth.local',
       'ALPTRAN23.accretivehealth.local',
       'ALPTRAN30.accretivehealth.local',
       'AHS-TRAN19.accretivehealth.local',
       'AHS-TRAN08.accretivehealth.local',
       'AHS-TRAN01.accretivehealth.local',
       'AHS-TRAN20.accretivehealth.local', 'AHS-A2RSAS03.extapp.local',
       'AHS-TRAN16.accretivehealth.local',
       

In [23]:
df_total.shape

(139734, 6)

In [24]:
chunks = [sub_df for _, sub_df in df_total.groupby("Server Name")]

In [25]:
"""
chunk_size = 10000
chunks = [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

"""

'\nchunk_size = 10000\nchunks = [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]\n\n'

In [26]:
# Create Automatic Folder in Desktop named Remaining Data
import os
from datetime import datetime

# Get the user's Downloads folder path
downloads_path = r"C:\Users\IN10052060\Desktop"

# Create a folder name with timestamp (or any custom name)
folder_name = "Remaining Data"
folder_path = os.path.join(downloads_path, folder_name)

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)


print(f"Folder created: {folder_path}")

Folder created: C:\Users\IN10052060\Desktop\Remaining Data


In [27]:
import os
from datetime import datetime

# Get the user's Downloads folder path
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")

# Create a folder name with timestamp (or any custom name)
folder_name = "SQL Query"
folder_path = os.path.join(downloads_path, folder_name)

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)


print(f"Folder created: {folder_path}")

Folder created: C:\Users\IN10052060\Downloads\SQL Query


In [28]:
import os
import glob

# Define your output folder path
output_dir = r'C:\Users\IN10052060\Downloads\SQL Query'

# ✅ Step 1: Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# ✅ Step 2: Delete all .txt files in the folder
for file in glob.glob(os.path.join(output_dir, '*.txt')):
    os.remove(file)
    #print(f"Deleted: {file}")

# print("Cleanup complete!")


In [29]:
sql_template  = """

SET NOCOUNT ON	



IF OBJECT_ID('tempdb..#Act') IS NOT NULL DROP TABLE #Act

create table #act(facilitycode char(4),encounterid varchar(50))
--declare @facility char(4)=right(db_name(),4)

declare @facility char(4)=right(db_name(),4)


--*********

-- INSERT_PLACEHOLDER

--***********

--select * from #act
IF OBJECT_ID('tempdb..#tmp') IS NOT NULL DROP TABLE #tmp
Select distinct  R.Facilitycode, R.EncounterID,r.id,
 --FirstName,LastName



CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,

pri.MajorPayerName CurrentMajoryPayer,
CurrentPayorPlanCode




Into #tmp	
From #act a

	 join dbo.Registrations(nolock) r on a.encounterid=r.EncounterID and r.facilitycode=a.facilitycode
	 inner join paymentssummary (nolock) PM on pm.RegistrationID=r.id 
	 left join (SELECT 
       --fpp.FacilityCode, 
       fpp.FacilityPlanCode,
       fpp.PayerPlanName,
       ---cd.CodeID AS MajorPayerID,
       cd.Code AS MajorPayerName
FROM 
       Accretive_FacilityPayerPlan fpp
       INNER JOIN Accretive_CodeDetail cd ON fpp.MajorPayerId = cd.CodeID  AND cd.CategoryID = 6

	   where fpp.FacilityCode=right(db_name(),4))Pri on CurrentPayorPlanCode=Pri.FacilityPlanCode
	 --left join tmpretro t on a.encounterid=t.PatientAccountNbr and a.facilitycode=t.facilitycode
	 --left join payments Payment

	IF OBJECT_ID('tempdb..#Activity') IS NOT NULL DROP TABLE #Activity
Select  R.Facilitycode, R.EncounterID,r.id, CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,
p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
u.UserID,
u.Username,
AA.[Name] ActivityCode,
CurrentMajoryPayer,
CurrentPayorPlanCode 

Into #Activity	
From #tmp r
 left join  dbo.ProcessLogs P With (NoLock) on r.id=P.Recordkey
	
	
	Left Join Accretive_Actions AA With (NoLock) on P.NextAction =AA.ID
	left join DNN.dbo.Users U With (NoLock) on P.CreatedBY=U.UserID
	--Where  NextAction is not NULL and NextAction <>0 and  AA.[Name] not like'%TB%'
	Order BY P.CreatedDateTime DESC


--- select * from #Activity

IF OBJECT_ID('tempdb..#tmtf') IS NOT NULL DROP TABLE #tmtf
 select Row_number() over(Partition by t.Encounterid order by createddatetime desc) sr,t.*

 into #tmtf
 from #Activity t




IF OBJECT_ID('tempdb..#fl') IS NOT NULL DROP TABLE #fl
 select r.*,Postdate 
 into #fl from #tmtf r
left join (select RegistrationID,max(postdate) Postdate from payments (nolock)
  where  TypeOfTransaction like ('A%')
   --and registrationid='6389156'
  group by RegistrationID) P12 on p12.RegistrationID=r.id 
 where sr=1

 IF OBJECT_ID('tempdb..#f') IS NOT NULL DROP TABLE #f
 select r.*,
 p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
UserID,
Username,
ActivityCode
into #f
from #tmp r
 left join #fl p on r.ID=p.ID

  select r.*
 ,act_date_2,Username2,ActivityCode2
 ,act_date_3,Username3,ActivityCode3
 from #f r
  left join (select id,CreatedDateTime act_date_2, Username Username2 ,ActivityCode  ActivityCode2 from #tmtf f where sr in('2')) f on f.id=r.id
  left join (select id,CreatedDateTime act_date_3, Username Username3 ,ActivityCode  ActivityCode3 from #tmtf f where sr in('3')) ff on ff.id=r.id


"""




sql_template = """

SET NOCOUNT ON	



IF OBJECT_ID('tempdb..#Act') IS NOT NULL DROP TABLE #Act

create table #act(facilitycode char(4),encounterid varchar(50))
--declare @facility char(4)=right(db_name(),4)

declare @facility char(4)=right(db_name(),4)


--*********

-- INSERT_PLACEHOLDER

--***********

--select * from #act
IF OBJECT_ID('tempdb..#tmp') IS NOT NULL DROP TABLE #tmp
Select distinct  R.Facilitycode, R.EncounterID,r.id,
 --FirstName,LastName



CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,

pri.MajorPayerName CurrentMajoryPayer,
CurrentPayorPlanCode




Into #tmp	
From #act a

	 join dbo.Registrations(nolock) r on a.encounterid=r.EncounterID and r.facilitycode=a.facilitycode
	 inner join paymentssummary (nolock) PM on pm.RegistrationID=r.id 
	 left join (SELECT 
       --fpp.FacilityCode, 
       fpp.FacilityPlanCode,
       fpp.PayerPlanName,
       ---cd.CodeID AS MajorPayerID,
       cd.Code AS MajorPayerName
FROM 
       Accretive_FacilityPayerPlan fpp
       INNER JOIN Accretive_CodeDetail cd ON fpp.MajorPayerId = cd.CodeID  AND cd.CategoryID = 6

	   where fpp.FacilityCode=right(db_name(),4))Pri on CurrentPayorPlanCode=Pri.FacilityPlanCode
	 --left join tmpretro t on a.encounterid=t.PatientAccountNbr and a.facilitycode=t.facilitycode
	 --left join payments Payment

	IF OBJECT_ID('tempdb..#Activity') IS NOT NULL DROP TABLE #Activity
Select  R.Facilitycode, R.EncounterID,r.id, CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,
p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
u.UserID,
u.Username,
AA.[Name] ActivityCode,
CurrentMajoryPayer,
CurrentPayorPlanCode 

Into #Activity	
From #tmp r
 left join  dbo.ProcessLogs P With (NoLock) on r.id=P.Recordkey
	
	
	Left Join Accretive_Actions AA With (NoLock) on P.NextAction =AA.ID
	left join DNN.dbo.Users U With (NoLock) on P.CreatedBY=U.UserID
	--Where  NextAction is not NULL and NextAction <>0 and  AA.[Name] not like'%TB%'
	Order BY P.CreatedDateTime DESC


--- select * from #tmp

IF OBJECT_ID('tempdb..#tmtf') IS NOT NULL DROP TABLE #tmtf
 select Row_number() over(Partition by t.Encounterid order by createddatetime desc) sr,t.*

 into #tmtf
 from #Activity t


IF OBJECT_ID('tempdb..#fl') IS NOT NULL DROP TABLE #fl
 select r.*,Postdate 
 into #fl from #tmtf r
left join (select RegistrationID,max(postdate) Postdate from payments (nolock)
  where  TypeOfTransaction like ('A%')
   --and registrationid='6389156'
  group by RegistrationID) P12 on p12.RegistrationID=r.id 
 where sr=1

 select r.*,
 p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
UserID,
Username,
ActivityCode

from #tmp r
 left join #fl p on r.ID=p.ID

"""



sql_template = """

SET NOCOUNT ON	



IF OBJECT_ID('tempdb..#Act') IS NOT NULL DROP TABLE #Act

create table #act(facilitycode char(4),encounterid varchar(50))
--declare @facility char(4)=right(db_name(),4)

declare @facility char(4)=right(db_name(),4)

--*********

-- INSERT_PLACEHOLDER
--***********

--select * from #act
IF OBJECT_ID('tempdb..#tmp') IS NOT NULL DROP TABLE #tmp
Select distinct  R.Facilitycode, R.EncounterID,r.id,
 --FirstName,LastName



CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,

pri.MajorPayerName CurrentMajoryPayer,
CurrentPayorPlanCode




Into #tmp	
From #act a

	 join dbo.Registrations(nolock) r on a.encounterid=r.EncounterID and r.facilitycode=a.facilitycode
	 inner join paymentssummary (nolock) PM on pm.RegistrationID=r.id 
	 left join (SELECT 
       --fpp.FacilityCode, 
       fpp.FacilityPlanCode,
       fpp.PayerPlanName,
       ---cd.CodeID AS MajorPayerID,
       cd.Code AS MajorPayerName
FROM 
       Accretive_FacilityPayerPlan fpp
       INNER JOIN Accretive_CodeDetail cd ON fpp.MajorPayerId = cd.CodeID  AND cd.CategoryID = 6

	   where fpp.FacilityCode=right(db_name(),4))Pri on CurrentPayorPlanCode=Pri.FacilityPlanCode
	 --left join tmpretro t on a.encounterid=t.PatientAccountNbr and a.facilitycode=t.facilitycode
	 --left join payments Payment

	IF OBJECT_ID('tempdb..#Activity') IS NOT NULL DROP TABLE #Activity
Select  R.Facilitycode, R.EncounterID,r.id, CurrentFinancialClass,EncounterInsuranceBalance,EncounterOpenBalance,EncounterPatientBalance,EncounterTotalCharges,
p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
u.UserID,
u.Username,
AA.[Name] ActivityCode,
CurrentMajoryPayer,
CurrentPayorPlanCode 

Into #Activity	
From #tmp r
 left join  dbo.ProcessLogs P With (NoLock) on r.id=P.Recordkey
	
	
	Left Join Accretive_Actions AA With (NoLock) on P.NextAction =AA.ID
	left join DNN.dbo.Users U With (NoLock) on P.CreatedBY=U.UserID
	Where  NextAction is not NULL and NextAction <>0 and  AA.[Name] not like'%TB%'
	Order BY P.CreatedDateTime DESC

--- select * from 

IF OBJECT_ID('tempdb..#tmtf') IS NOT NULL DROP TABLE #tmtf
 select Row_number() over(Partition by t.Encounterid order by act.createddatetime desc) sr,t.*

 into #tmtf
 from #Activity t
left join #Activity act (nolock) on act.EncounterID=t.EncounterID
left join  TempClaimsPayments (nolock) TC on TC.REGISTRATIONID=t.id


IF OBJECT_ID('tempdb..#fl') IS NOT NULL DROP TABLE #fl
 select r.*,Postdate 
 into #fl from #tmtf r
left join (select RegistrationID,max(postdate) Postdate from payments (nolock)
  where  TypeOfTransaction like ('A%')
   --and registrationid='6389156'
  group by RegistrationID) P12 on p12.RegistrationID=r.id 
 where sr=1

 select r.*,
 p.CreatedDateTime
,p.ScheduleDateTime,
p.Createdby,
p.Note,
UserID,
Username,
ActivityCode

from #tmp r
 left join #fl p on r.ID=p.ID

"""





In [30]:

for i, chunk in enumerate(chunks, start=1):
    series = chunk['Query Code'].fillna('')  # replace NaN with empty strings
    series = series.astype(str)

    # Optional: remove any literal 'nan' strings that may appear due to prior casts
    series = series.map(lambda s: '' if s.lower() == 'nan' else s)

    insert_lines = "\n".join(series)

    full_sql = sql_template.replace('-- INSERT_PLACEHOLDER', insert_lines)

    file_path = os.path.join(output_dir, f'sql_chunk_{i}.sql')
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(full_sql)


## Optimized Manner

In [31]:
import os
import re
import pandas as pd
import pyodbc
from datetime import datetime
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import winsound
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# --- Config ---
DB_STRUCTURE_PATH = r'C:\Users\IN10052060\Desktop\Python\SqlDBStructure.xlsx'
MASTER_FILE_PATH = r'C:\Users\IN10052060\Desktop\Python\Mapping.xlsx'
SQL_FOLDER = r"C:\Users\IN10052060\Downloads\SQL Query"
OUTPUT_DIR = r"C:\Users\IN10052060\Desktop\Remaining Data"

# --- Step 1: Load DB structure ---
db_file = load_workbook(DB_STRUCTURE_PATH)
db_sheet = db_file['All Sites']

targets = []
for i in range(2, db_sheet.max_row + 1):
    facility = db_sheet.cell(row=i, column=1).value
    database = db_sheet.cell(row=i, column=2).value
    server = db_sheet.cell(row=i, column=3).value
    if server and database:
        targets.append((i, facility, database, server))

# --- Encoding-safe SQL reader ---
def read_sql_file(path):
    encodings = ["utf-8", "cp1252", "latin-1"]

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    # Last resort: ignore errors
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

# --- Cleaning helpers ---
def clean_illegal_characters(value):
    if isinstance(value, str):
        return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x84\x86-\x9f]', '', value)
    return value

def clean_dataframe(df):
    df_cleaned = df.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].apply(clean_illegal_characters)
    df_cleaned.columns = df_cleaned.columns.str.strip()
    df_cleaned.drop_duplicates(inplace=True)
    return df_cleaned

# --- SQL execution ---
def fetch_data(row_num, facility, database, server, query):
    try:
        print(f"Connecting to {database} on {server}...")
        start = datetime.now()

        conn = pyodbc.connect(
            f"Driver={{SQL Server}};Server={server};Database={database};Trusted_Connection=yes;"
        )
        df = pd.read_sql(query, conn)
        conn.close()

        duration = (datetime.now() - start).total_seconds()
        df_cleaned = clean_dataframe(df)

        return (row_num, facility, df_cleaned, len(df_cleaned), duration, "Success")

    except Exception as e:
        print(f"❌ Error for {facility}: {e}")
        return (row_num, facility, None, 0, 0, f"Error: {str(e)}")

# --- Extract facility codes from SQL file ---
def get_facilities_from_sql(query_text):
    return re.findall(r"insert into #act values\('([A-Z0-9]+)'", query_text)

# --- MAIN EXECUTION LOOP ---
sql_files = [f for f in os.listdir(SQL_FOLDER) if f.lower().endswith((".sql", ".txt"))]

for sql_file in sql_files:
    sql_path = os.path.join(SQL_FOLDER, sql_file)
    query = read_sql_file(sql_path)

    print(f"\n🔍 Running query from: {sql_file}")

    facilities_in_file = get_facilities_from_sql(query)
    if not facilities_in_file:
        print(f"⚠️ No facility codes found in {sql_file}, skipping.")
        continue

    subset_targets = [t for t in targets if t[1] in facilities_in_file]
    if not subset_targets:
        print(f"⚠️ No matching servers found for facilities {facilities_in_file}, skipping.")
        continue

    master_file = Workbook()
    log_sheet = master_file.active
    log_sheet.title = "Log"
    raw_data_sheet = master_file.create_sheet("RawData")

    log_sheet.append(["Facility", "Status", "Row Count", "Duration (s)"])

    dfs = []
    header_written = False

    for args in subset_targets:
        row_num, facility, database, server = args

        row_num, facility, df, row_count, duration, status = fetch_data(
            row_num, facility, database, server, query
        )

        log_sheet.append([facility, status, row_count, round(duration, 2)])

        if df is not None and not df.empty:
            if not header_written:
                for row in dataframe_to_rows(df, index=False, header=True):
                    raw_data_sheet.append(row)
                header_written = True
            else:
                for row in dataframe_to_rows(df, index=False, header=False):
                    raw_data_sheet.append(row)

            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base_name = os.path.splitext(sql_file)[0]
    output_file = os.path.join(OUTPUT_DIR, f"{base_name}_{timestamp}.xlsx")

    master_file.save(output_file)

    print(f"✅ Saved result for {sql_file}: {output_file}")
    print(f"📊 Combined shape: {combined_df.shape}\n")

winsound.MessageBeep()



🔍 Running query from: sql_chunk_1.sql
Connecting to TranE120 on AHS-A2ATRN01.extapp.local...
Connecting to TranE122 on AHS-A2ATRN01.extapp.local...
Connecting to TranE129 on AHS-A2ATRN01.extapp.local...
Connecting to TranE130 on AHS-A2ATRN01.extapp.local...
Connecting to TranE132 on AHS-A2ATRN01.extapp.local...
Connecting to TranE154 on AHS-A2ATRN01.extapp.local...
Connecting to TranA279 on AHS-A2ATRN01.extapp.local...
✅ Saved result for sql_chunk_1.sql: C:\Users\IN10052060\Desktop\Remaining Data\sql_chunk_1_20260619-202832.xlsx
📊 Combined shape: (1573, 23)


🔍 Running query from: sql_chunk_10.sql
Connecting to TranBYFL on AHS-TRAN12.accretivehealth.local...
Connecting to TranLCWA on AHS-TRAN12.accretivehealth.local...
Connecting to TranLHSC on AHS-TRAN12.accretivehealth.local...
Connecting to TranLLMT on AHS-TRAN12.accretivehealth.local...
Connecting to TranLMNC on AHS-TRAN12.accretivehealth.local...
Connecting to TranLMPA on AHS-TRAN12.accretivehealth.local...
Connecting to TranLNPA

C:\Users\local_IN10052060\Temp\ipykernel_40808\4055168822.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


✅ Saved result for sql_chunk_23.sql: C:\Users\IN10052060\Desktop\Remaining Data\sql_chunk_23_20260619-204037.xlsx
📊 Combined shape: (8134, 23)


🔍 Running query from: sql_chunk_24.sql
Connecting to TranCHMI on ALPTRAN25.accretivehealth.local...
Connecting to TranSTIN on ALPTRAN25.accretivehealth.local...
Connecting to TranSVIN on ALPTRAN25.accretivehealth.local...
Connecting to TranVAIN on ALPTRAN25.accretivehealth.local...
Connecting to TranVCIN on ALPTRAN25.accretivehealth.local...
Connecting to TranVFIN on ALPTRAN25.accretivehealth.local...
✅ Saved result for sql_chunk_24.sql: C:\Users\IN10052060\Desktop\Remaining Data\sql_chunk_24_20260619-204157.xlsx
📊 Combined shape: (8332, 23)


🔍 Running query from: sql_chunk_25.sql
Connecting to TranVHIN on ALPTRAN26.accretivehealth.local...
Connecting to TranVJIN on ALPTRAN26.accretivehealth.local...
Connecting to TranVKIN on ALPTRAN26.accretivehealth.local...
Connecting to TranVMIN on ALPTRAN26.accretivehealth.local...
Connecting to TranVRIN

# Agrregate the data and Planning to run the balance 2nd Time

In [32]:
import os
import pandas as pd

# --- CONFIGURATION ---
EXCEL_FOLDER = r"C:\Users\IN10052060\Desktop\Remaining Data"  # Update to your folder path
combined_df = pd.DataFrame()

# --- LOOP THROUGH EXCEL FILES ---
excel_files = [f for f in os.listdir(EXCEL_FOLDER) if f.endswith(".xlsx")]

print(f"Found {len(excel_files)} Excel files:")
for file in excel_files:
    file_path = os.path.join(EXCEL_FOLDER, file)
    try:
        # Read the 'RawData' sheet (adjust if needed)
        df = pd.read_excel(file_path, sheet_name="RawData")
        df["SourceFile"] = file  # Optional: track origin
        combined_df = pd.concat([combined_df, df], ignore_index=True)
        print(f"✅ Loaded: {file} ({df.shape[0]} rows)")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

# --- PREVIEW RESULT ---
print(f"\n📊 Combined shape: {combined_df.shape}")
print(combined_df.head())


Found 36 Excel files:
✅ Loaded: sql_chunk_10_20260619-203037.xlsx (9507 rows)
✅ Loaded: sql_chunk_11_20260619-203110.xlsx (6512 rows)
✅ Loaded: sql_chunk_12_20260619-203205.xlsx (6564 rows)
✅ Loaded: sql_chunk_13_20260619-203253.xlsx (4765 rows)
✅ Loaded: sql_chunk_14_20260619-203255.xlsx (5 rows)
✅ Loaded: sql_chunk_15_20260619-203408.xlsx (6383 rows)
✅ Loaded: sql_chunk_16_20260619-203438.xlsx (4136 rows)
✅ Loaded: sql_chunk_17_20260619-203533.xlsx (10670 rows)
✅ Loaded: sql_chunk_18_20260619-203707.xlsx (5753 rows)
✅ Loaded: sql_chunk_19_20260619-203717.xlsx (1914 rows)
✅ Loaded: sql_chunk_1_20260619-202832.xlsx (1573 rows)
✅ Loaded: sql_chunk_20_20260619-203804.xlsx (3167 rows)
✅ Loaded: sql_chunk_21_20260619-203833.xlsx (3987 rows)
✅ Loaded: sql_chunk_22_20260619-203920.xlsx (6052 rows)
✅ Loaded: sql_chunk_23_20260619-204037.xlsx (8134 rows)
✅ Loaded: sql_chunk_24_20260619-204157.xlsx (8332 rows)
✅ Loaded: sql_chunk_25_20260619-204335.xlsx (6245 rows)
✅ Loaded: sql_chunk_26_202606

In [33]:
combined_df.shape

(137149, 24)

In [34]:
combined_df.to_excel(os.path.join(EXCEL_FOLDER, "Combined_Output.xlsx"), index=False)


In [35]:
import os
import time
import pandas as pd
from openpyxl import Workbook, load_workbook

start_time = time.time()

file_path = r"C:\Users\IN10052060\OneDrive - R1\Target Balance support\Balance Output.xlsx"
#file_path = r"C:\Users\IN10052060\Downloads\Balance Output.xlsx"

# Step 1: Check if file exists, delete if yes
if os.path.exists(file_path):
    os.remove(file_path)
    print("🗑️ Existing file deleted.")

# Step 2: Create a fresh Excel file
wb = Workbook()
wb.save(file_path)
print("📂 New Excel file created.")

# Step 3: Export your DataFrame to the new file
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a') as writer:
    combined_df.to_excel(writer, sheet_name='UniqeData', index=False)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"✅ Data successfully written to the sheet in {elapsed_time:.2f} seconds")

🗑️ Existing file deleted.
📂 New Excel file created.
✅ Data successfully written to the sheet in 60.49 seconds


In [36]:
combined_df.shape

(137149, 24)

# End of Code